# GCP Pipeline — Same Queries, Infinite Scale

**The Analytics Pipeline notebook ran on Pandas. This one runs on GCS + BigQuery.**

Same data. Same questions. But BigQuery handles 50 million rows without breaking a sweat.

This notebook is meant to be run cell-by-cell in a terminal or Cloud Shell — the `!` prefix runs shell commands from Jupyter/Colab.

---

## Step 0: GCP Setup (One-Time)

Run these once. Skip if already done.

In [ ]:
# === CONFIGURATION — Change these ===
PROJECT_ID = "data-pipeline-project-490820"
BUCKET_NAME = f"de-accelerator-{PROJECT_ID}"  # Globally unique
DATASET_NAME = "call_center_raw"
LOCATION = "us-central1"

print(f"Project:  {PROJECT_ID}")
print(f"Bucket:   gs://{BUCKET_NAME}/")
print(f"Dataset:  {DATASET_NAME}")
print(f"Location: {LOCATION}")

In [ ]:
# Authenticate (if running locally — Colab handles this differently)
# Uncomment the next line if you need to login:
# !gcloud auth login

# Set the project
!gcloud config set project {PROJECT_ID}

In [ ]:
# Enable required APIs (only needed once per project)
!gcloud services enable bigquery.googleapis.com
!gcloud services enable storage.googleapis.com
print("APIs enabled.")

In [ ]:
# Create a storage bucket
# Bucket names must be globally unique across all of GCP
!gcloud storage buckets create gs://{BUCKET_NAME}/ --location={LOCATION} 2>&1 || echo "Bucket may already exist — that is fine."

# Verify
!gcloud storage ls gs://{BUCKET_NAME}/ 2>&1 | head -5
print(f"\nBucket ready: gs://{BUCKET_NAME}/")

---

## Step 1: BRONZE — Upload Raw Data to GCS

This is the cloud equivalent of "loading a CSV file." Instead of reading from your laptop's disk, the data lives in GCS (Google Cloud Storage) — accessible from anywhere, scalable to petabytes.

In production, data would land here automatically (API writes, file drops, streaming). For now, we upload manually.

In [ ]:
import os

# Find the data directory
DATA_DIR = "../data" if os.path.exists("../data/calls.json") else os.path.expanduser("~/Downloads/data-engineer-accelerator/data")
print(f"Data source: {DATA_DIR}")
print(f"Files: {os.listdir(DATA_DIR)}")

In [ ]:
# Upload all data files to the Bronze layer
# Bronze = raw, untouched. Same principle as the Pandas notebook.

!gcloud storage cp {DATA_DIR}/campaigns.csv gs://{BUCKET_NAME}/bronze/
!gcloud storage cp {DATA_DIR}/products.csv gs://{BUCKET_NAME}/bronze/
!gcloud storage cp {DATA_DIR}/orders.csv gs://{BUCKET_NAME}/bronze/
!gcloud storage cp {DATA_DIR}/calls.json gs://{BUCKET_NAME}/bronze/

print("\nBronze layer loaded.")

In [ ]:
# Verify — list everything in the bronze folder
!gcloud storage ls gs://{BUCKET_NAME}/bronze/

---

## Step 2: Load into BigQuery

BigQuery is Google's serverless data warehouse. No servers to manage. You pay per query (bytes scanned). It can scan terabytes in seconds.

This is the cloud equivalent of `pd.read_csv()` — but it scales to billions of rows.

In [ ]:
# Create a BigQuery dataset (like a schema/database — a container for tables)
!bq mk --dataset --location={LOCATION} {PROJECT_ID}:{DATASET_NAME} 2>&1 || echo "Dataset may already exist."

print(f"Dataset ready: {PROJECT_ID}:{DATASET_NAME}")

In [ ]:
# Load CSV files into BigQuery tables
# --autodetect lets BigQuery infer column names and types from the CSV header

!bq load --autodetect --replace --source_format=CSV \
  {DATASET_NAME}.campaigns \
  gs://{BUCKET_NAME}/bronze/campaigns.csv

print("campaigns loaded.")

In [ ]:
!bq load --autodetect --replace --source_format=CSV \
  {DATASET_NAME}.products \
  gs://{BUCKET_NAME}/bronze/products.csv

print("products loaded.")

In [ ]:
!bq load --autodetect --replace --source_format=CSV \
  {DATASET_NAME}.orders \
  gs://{BUCKET_NAME}/bronze/orders.csv

print("orders loaded.")

In [ ]:
# Load the calls JSON
# calls.json is newline-delimited JSON (NDJSON) — each line is one record
!bq load --autodetect --replace --source_format=NEWLINE_DELIMITED_JSON \
  {DATASET_NAME}.calls \
  gs://{BUCKET_NAME}/bronze/calls.json

print("calls loaded.")

In [ ]:
# Verify — list all tables in the dataset
!bq ls {DATASET_NAME}

In [ ]:
# Preview the calls table — does it look right?
!bq head -n 3 {DATASET_NAME}.calls

---

## Step 3: Query BigQuery — Same Questions, Cloud Scale

These are the **same queries** from the Analytics Pipeline notebook. There, they ran on Pandas. Here, they run on BigQuery SQL.

On 500 rows, both are fast. On 50 million rows, Pandas crashes. BigQuery finishes in seconds.

In [ ]:
# Query 1: Call count and average duration by campaign
# In Pandas: fact_calls.groupby(["campaign_name"]).agg(avg_duration=("duration", "mean"), ...)
# In BigQuery: SQL

!bq query --use_legacy_sql=false '
SELECT
    c.campaign_name,
    c.channel,
    COUNT(*) AS total_calls,
    ROUND(AVG(calls.duration), 1) AS avg_duration
FROM `{DATASET_NAME}.calls` AS calls
JOIN `{DATASET_NAME}.campaigns` AS c
    ON CAST(calls.dnis AS STRING) = CAST(c.dnis AS STRING)
GROUP BY c.campaign_name, c.channel
ORDER BY avg_duration DESC
'

In [ ]:
# Query 2: Disposition (outcome) distribution
# In Pandas: calls["disposition"].value_counts()
# In BigQuery: GROUP BY

!bq query --use_legacy_sql=false '
SELECT
    disposition,
    COUNT(*) AS count
FROM `{DATASET_NAME}.calls`
GROUP BY disposition
ORDER BY count DESC
'

In [ ]:
# Query 3: Calls by day of week
# In Pandas: needed date extraction + groupby
# In BigQuery: FORMAT_DATE gives us the day name directly

!bq query --use_legacy_sql=false '
SELECT
    FORMAT_DATE("%A", date) AS day_name,
    CASE WHEN EXTRACT(DAYOFWEEK FROM date) IN (1, 7) THEN true ELSE false END AS is_weekend,
    COUNT(*) AS total_calls
FROM `{DATASET_NAME}.calls`
WHERE date IS NOT NULL
GROUP BY day_name, is_weekend
ORDER BY total_calls DESC
'

In [ ]:
# Query 4: VA vs Live channel comparison

!bq query --use_legacy_sql=false '
SELECT
    c.channel,
    COUNT(*) AS total_calls,
    ROUND(AVG(calls.duration), 1) AS avg_duration
FROM `{DATASET_NAME}.calls` AS calls
JOIN `{DATASET_NAME}.campaigns` AS c
    ON CAST(calls.dnis AS STRING) = CAST(c.dnis AS STRING)
GROUP BY c.channel
'

In [ ]:
# Query 5: Calls by hour of day (time-of-day analysis)
# Useful for staffing decisions — when do we need more agents?

!bq query --use_legacy_sql=false '
SELECT
    EXTRACT(HOUR FROM start_time) AS hour_of_day,
    COUNT(*) AS total_calls,
    ROUND(AVG(duration), 1) AS avg_duration
FROM `{DATASET_NAME}.calls`
WHERE start_time IS NOT NULL
GROUP BY hour_of_day
ORDER BY hour_of_day
'

In [ ]:
# Query 6: Orders and revenue by campaign

!bq query --use_legacy_sql=false '
SELECT
    c.campaign_name,
    COUNT(DISTINCT o.order_id) AS order_count,
    ROUND(SUM(o.total), 2) AS total_revenue,
    ROUND(AVG(o.total), 2) AS avg_order_value
FROM `{DATASET_NAME}.orders` AS o
JOIN `{DATASET_NAME}.calls` AS calls
    ON o.call_id = calls.call_id
JOIN `{DATASET_NAME}.campaigns` AS c
    ON CAST(calls.dnis AS STRING) = CAST(c.dnis AS STRING)
GROUP BY c.campaign_name
ORDER BY total_revenue DESC
'

---

## What Just Happened — Pandas vs BigQuery

| | Pandas (Analytics_Pipeline notebook) | BigQuery (this notebook) |
|---|---|---|
| **Data location** | Your laptop's RAM | Google Cloud — distributed across servers |
| **Query language** | Python (`.groupby().agg()`) | SQL |
| **500 rows** | 2 seconds | 2 seconds |
| **50 million rows** | Crashes (out of memory) | 3-5 seconds |
| **500 million rows** | Impossible | 10-15 seconds |
| **Cost** | Free (your hardware) | ~$0.005 per query on 500 rows. ~$2.50 per query on 50M rows. |
| **Scheduling** | Run manually | Cloud Composer (Airflow) runs it every night |
| **Collaboration** | Share the notebook | Anyone with BigQuery access runs the same queries |

**The pipeline pattern is identical.** Bronze → Silver → Gold → Queries. Only the execution engine changed.

---

## Next Steps

1. **Build the star schema in BigQuery** — `CREATE TABLE` with surrogate keys, dimension tables, partitioning
2. **Schedule with Cloud Composer** — DAG that runs Bronze → Silver → Gold every night
3. **Add data quality checks** — detect duplicates, nulls, timezone issues before they reach Gold
4. **Connect a dashboard** — Looker Studio or Data Studio reads from the Gold tables

In [ ]:
# Cleanup — uncomment to delete resources when done
# !bq rm -r -f {DATASET_NAME}
# !gcloud storage rm -r gs://{BUCKET_NAME}/
# print("Resources deleted.")